# Lab 4 — Notebook 01: Regression — Predicting Apartment Rent

**Course:** Artificial Intelligence with Python  
**Duration:** 45 minutes (core route)  
**Prerequisites:** Lab 1 and Lab 2 (data cleaning, feature preprocessing, and train/test splitting)  
**Files:** `data/apartments_corrupted.csv`  
**Tools:** Pandas, NumPy, Matplotlib, scikit-learn

## Learning objectives

After completing this notebook, you should be able to:

1. Describe **regression** as predicting a continuous value.
2. Reconstruct the preprocessing workflow from Lab 2 in a new notebook.
3. Train a **baseline** and a **linear regression** model using `.fit()`.
4. Make predictions with `.predict()` and evaluate them using **MAE**, **RMSE**, and **R²**.
5. Interpret prediction errors and make a prediction for a new apartment.

**Central question:** Can we predict an apartment's monthly rent from its characteristics?

> **Student instructions:** Run the notebook from top to bottom. Complete each **TODO** exercise and write short answers in the Markdown cells. Worked examples are executable without the exercise solutions. The dataset is **synthetic**, not real rental-market data.


**Route:** Complete the worked examples and core reflections. Section times include their exercises: 5 + 5 + 10 + 12 + 10 + 3 = **45 minutes**. The extra apartment challenge is optional. Restart and Run All after completing your work; no TODO solution is required by later examples.


## 1. Reload the familiar apartment data (5 min, including Exercise 1)

**Core.** Reuse Lab 2's supplied cleaning function below. Read its comments and run it; you do not need to rewrite it. It applies fixed rules, excludes ambiguous duplicate IDs and unusable targets, and leaves missing features for the training-only imputer. Positive outliers remain unless there is evidence of an error.

Today's new work is the complete **preprocessing + estimator** pipeline and interpretation of its predictions. Keep `data/apartments_corrupted.csv` beside the notebook's `data/` directory structure: run this notebook inside `lab04/`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

csv_path = Path("data/apartments_corrupted.csv")
if not csv_path.is_file():
    raise FileNotFoundError(
        "Cannot find data/apartments_corrupted.csv. Open JupyterLab in the lab04 folder "
        "or use the repository's lab04 directory as your working directory."
    )

raw = pd.read_csv(csv_path)
print("Raw dataset:", raw.shape)
display(raw.head())

In [ ]:
def clean_apartments(raw_df):
    """Apply fixed, row-wise cleaning rules; do not learn train/test statistics."""
    df = raw_df.drop_duplicates().copy()

    # Same ID with different values: set aside for manual review.
    conflicting_ids = df.loc[
        df["apartment_id"].duplicated(keep=False), "apartment_id"
    ].unique()
    df = df.loc[~df["apartment_id"].isin(conflicting_ids)].copy()

    numeric_cols = [
        "area_m2", "rooms", "building_age_years",
        "distance_to_center_km", "monthly_rent_eur"
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    invalid = {
        "area_m2": df["area_m2"] <= 0,
        "rooms": df["rooms"] < 1,
        "building_age_years": df["building_age_years"] < 0,
        "distance_to_center_km": df["distance_to_center_km"] < 0,
    }
    for col, mask in invalid.items():
        df.loc[mask, col] = np.nan

    df["district"] = df["district"].astype("string").str.strip().str.title()
    df["energy_rating"] = df["energy_rating"].astype("string").str.strip().str.upper()
    balcony_text = df["balcony"].astype("string").str.strip().str.lower()
    df["balcony"] = balcony_text.map({
        "yes": "yes", "y": "yes", "true": "yes",
        "no": "no", "n": "no", "false": "no"
    })

    # Use np.nan for compatibility with scikit-learn's categorical imputer.
    for col in ["district", "balcony", "energy_rating"]:
        df[col] = df[col].astype(object).where(df[col].notna(), np.nan)

    df = df.loc[
        df["monthly_rent_eur"].notna() & (df["monthly_rent_eur"] > 0)
    ].copy()
    return df

df = clean_apartments(raw)
print("Usable rows:", len(df))
print("Missing features (still to be imputed):")
print(df.isna().sum()[df.isna().sum() > 0])

### Core — Check the boundary between features and target

Count missing `area_m2` values. Why must `monthly_rent_eur` be excluded from the input features?


In [ ]:
# TODO: count missing area_m2 values in df
# TODO: explain target leakage in the Markdown cell below


**Your explanation:** *TODO: Why would including the target among the features be data leakage?*

## 2. Separate features and target; reserve test data (5 min, including reflection)

In supervised learning, the model learns a relationship between **input features (`X`)** and a known **target (`y`)**.

- **Inputs:** area, number of rooms, age, distance to city center, district, balcony, energy rating.
- **Target:** `monthly_rent_eur`, a continuous value, so this is a **regression** task.
- `apartment_id` is an identifier, not a useful feature.

We hold out 20% of the observations for a final evaluation. From now on, **we must not fit preprocessing or choose models by looking at the test set**.


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["apartment_id", "monthly_rent_eur"])
y = df["monthly_rent_eur"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)
assert X_train.index.intersection(X_test.index).empty

### Exercise 2 — Explain the split

Why do we use `random_state=42`? Why should we avoid repeatedly evaluating candidate models on `X_test` and choosing the best one?

**Your answer:** *TODO: Write 2–3 sentences.*

## 3. Build a complete preprocessing + model pipeline (10 min, including prediction exercise)

In Lab 2, the last step was a **preprocessing pipeline**. Now we attach a **regression estimator** to it. This means calling `.fit(X_train, y_train)` both learns preprocessing parameters **from the training features** and fits the regression model using the prepared features.

For numerical inputs, we impute missing values with the **training median** and standardize them. For categories, we impute the most frequent category and use one-hot encoding.

`OneHotEncoder(handle_unknown="ignore")` lets us predict for an apartment in a previously unseen district.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression

numeric_features = [
    "area_m2", "rooms", "building_age_years", "distance_to_center_km"
]
categorical_features = ["district", "balcony", "energy_rating"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

rent_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("regression", LinearRegression()),
])
print(rent_model)

### What does `.fit()` do?

- **Preprocessor** learns numeric medians, scaling statistics, and category encodings from `X_train`.
- **Linear regression** learns coefficients that relate the transformed input features to rental prices in `y_train`.
- `.predict(X_test)` applies the **same preprocessing learned during training** and returns rent estimates in euros.

The modeling pipeline makes it harder to accidentally fit preprocessing separately on the test set.

In [ ]:
rent_model.fit(X_train, y_train)

train_predictions = rent_model.predict(X_train)
test_predictions = rent_model.predict(X_test)

preview = pd.DataFrame({
    "actual_rent_eur": y_test.iloc[:8].to_numpy(),
    "predicted_rent_eur": np.round(test_predictions[:8], 2),
})
display(preview)

### Exercise 3 — Inspect individual predictions

1. For the first eight test apartments, calculate **prediction error = predicted − actual** in euros.
2. Identify the example with the largest **absolute error**.
3. Is the model's prediction always exactly equal to the actual rent? Why would that be unrealistic?

**Hint:** You can use `preview`, `abs()`, and `.idxmax()`.

In [ ]:
# TODO: create an error_eur column in preview
# TODO: find the row with the largest absolute error
# TODO: explain the result in the next Markdown cell

**Your observation:** *TODO: Write 1–2 sentences.*

## 4. Establish a baseline and evaluate regression (12 min, including interpretation)

Before asking whether a model is good, we need a point of comparison. A **baseline** is a deliberately simple prediction strategy.

`DummyRegressor(strategy="median")` predicts the **median training rent for every apartment**. It learns nothing about area or location.

We will compare it with linear regression using three metrics:

- **MAE (Mean Absolute Error):** average absolute error, measured in euros; lower is better.
- **RMSE (Root Mean Squared Error):** also measured in euros; penalizes large errors more; lower is better.
- **R² (coefficient of determination):** 1 is perfect, 0 matches predicting the mean of the evaluation targets, and negative values mean worse than that mean baseline. The **median dummy** is chosen for its resistance to outliers and is not necessarily R² = 0.

**Important:** We compare these two preselected examples on the held-out test set for learning purposes. In a real project, choose features, hyperparameters, and the final model using *training/validation data only*; reserve the test set for one final evaluation.


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

def regression_metrics(actual, predicted):
    return {
        "MAE (€)": mean_absolute_error(actual, predicted),
        "RMSE (€)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    }

results = pd.DataFrame({
    "Median baseline": regression_metrics(y_test, baseline_predictions),
    "Linear regression": regression_metrics(y_test, test_predictions),
}).T

display(results.round(3))

### Exercise 4 — Interpret model quality

1. Which approach has a lower MAE?
2. What does its MAE mean in **euros per monthly prediction**?
3. Which metric places greater emphasis on very large errors: MAE or RMSE?
4. Suppose your MAE is €150. Does that mean **every** apartment estimate is off by exactly €150?

**Your answers:** *TODO: Give brief explanations, not just metric names.*

## 5. Visualize predictions and make a new estimate (10 min, including plot interpretation and new prediction)

Plots often reveal model behavior that a single metric hides. We will inspect:

1. **Actual vs. predicted rent:** ideal predictions lie on the dashed diagonal.
2. **Residuals:** `actual − predicted` should ideally show no obvious pattern; outliers are easier to spot.

Plots and metrics are shown on the test set here **only for illustration of evaluation**. Do not change the model repeatedly to improve these results.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(y_test, test_predictions, alpha=0.65)
lo = min(float(y_test.min()), float(test_predictions.min()))
hi = max(float(y_test.max()), float(test_predictions.max()))
ax.plot([lo, hi], [lo, hi], "k--", label="Perfect prediction")
ax.set(xlabel="Actual monthly rent (€)", ylabel="Predicted monthly rent (€)",
       title="Linear regression: Actual vs predicted")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test.to_numpy() - test_predictions

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(test_predictions, residuals, alpha=0.65)
ax.axhline(0, color="black", linestyle="--")
ax.set(xlabel="Predicted monthly rent (€)", ylabel="Residual (actual − predicted) (€)",
       title="Residual plot")
plt.tight_layout()
plt.show()

### Exercise 5 — Read the plots

1. Name one apartment with a particularly large **absolute residual**.
2. Does the scatter plot show a general relationship between actual and predicted rent?
3. Why might a linear model fail to represent some apartment prices?

**Your answers:** *TODO: Write 2–4 sentences.*

### Predict a newly listed apartment

We use **the already trained pipeline**, not a new fitted scaler or encoder. The following example is purely illustrative and includes a district not present in the training data.

In [ ]:
new_apartment = pd.DataFrame([{
    "area_m2": 75.0,
    "rooms": 3,
    "building_age_years": 15.0,
    "distance_to_center_km": 4.0,
    "district": "Harbor",       # unseen category — handled by the encoder
    "balcony": "yes",
    "energy_rating": "B",
}])

predicted_rent = rent_model.predict(new_apartment)[0]
print(f"Estimated monthly rent: €{predicted_rent:,.2f}")
print("This estimate is a model output based on synthetic teaching data.")

## Extension — Make another prediction (optional, 5 min)

Create an additional apartment, change at least two characteristics, and predict with the **existing fitted** pipeline. Explain the difference; do not refit or tune using the test results. Skip this extension if time is short. Cross-validation is a required topic in Notebook 03.


In [ ]:
# TODO: define another_apartment with the same input columns as new_apartment
# TODO: call rent_model.predict(another_apartment)
# TODO: compare the new estimate with predicted_rent

**Your interpretation:** *TODO: Write 2–3 sentences about the difference in predicted rents.*

## Core wrap-up (3 min)

**End-to-end workflow:** raw CSV → deterministic cleaning → define features and target → train/test split → fit preprocessing + regression on training data → predict → evaluate.

### Self-check

- [ ] I can explain why predicting rent is a **regression** problem.
- [ ] I can call `fit()` and `predict()` on a scikit-learn pipeline.
- [ ] I can explain why a **baseline** is useful.
- [ ] I can distinguish **MAE**, **RMSE**, and **R²**.
- [ ] I can read an actual-vs-predicted plot and a residual plot.
- [ ] I can predict for a new record without refitting preprocessing.

**Coming next:** Notebook 02 uses a larger **handwritten digits** dataset (1,797 samples; 10 digit classes) to introduce **classification** with k-nearest neighbors, logistic regression, and decision trees.
